In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# ── Apply all Task 2 cleaning steps ──
df = pd.read_csv('churnguard_data.csv')
df = df.drop('customerID', axis=1)
df = df.drop_duplicates()
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

contract_map = {
    'Month-to-month': 'Month-to-month', 'month to month': 'Month-to-month',
    'month-to-month': 'Month-to-month', 'Monthly': 'Month-to-month',
    'One year': 'One year', 'One Year': 'One year',
    'one year': 'One year', '1 year': 'One year',
    'Two year': 'Two year', 'Two Year': 'Two year',
    'two year': 'Two year', '2 year': 'Two year',
}
df['Contract'] = df['Contract'].map(contract_map)

internet_map = {
    'DSL': 'DSL', 'dsl': 'DSL',
    'Fiber optic': 'Fiber optic', 'Fibre optic': 'Fiber optic',
    'FiberOptic': 'Fiber optic', 'fiber optic': 'Fiber optic',
    'No': 'No', 'None': 'No', 'none': 'No',
}
df['InternetService'] = df['InternetService'].map(internet_map)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] > 0]
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(int(df['tenure'].median()))

# ── Encode target ──
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Use only 5 features ──
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 
            'SeniorCitizen', 'Contract']

# ── Encode Contract as ordinal ──
contract_ordinal = {
    'Month-to-month': 0,
    'One year':       1,
    'Two year':       2,
}
df['Contract'] = df['Contract'].map(contract_ordinal)

X = df[features]
y = df['Churn']

# ── Train on full dataset ──
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# ── Collect user inputs ──
print("\n--- Customer Churn Prediction ---")
tenure         = int(input("Enter tenure (months): "))
monthly        = float(input("Enter Monthly Charges: "))
total          = float(input("Enter Total Charges: "))
senior         = int(input("Senior Citizen? (1 = Yes, 0 = No): "))
contract       = int(input("Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): "))

# ── Predict ──
customer = pd.DataFrame([[tenure, monthly, total, senior, contract]],
                         columns=features)
prediction = model.predict(customer)[0]

if prediction == 1:
    print("\nPrediction: This customer is likely to CHURN.")
else:
    print("\nPrediction: This customer is likely to STAY.")


--- Customer Churn Prediction ---

Prediction: This customer is likely to STAY.


# Task 4 — Build a Prediction Script (ChurnGuard)

## Goal
Retrain model on full cleaned dataset and build interactive tool
for business team to predict churn for any customer instantly.

## Key Difference from Task 3
Task 3 → train/test split → evaluate model performance
Task 4 → train on FULL dataset → no split needed
(goal is prediction tool, not evaluation)

## Features Used (5 only)
tenure          → months with company (integer)
MonthlyCharges  → monthly bill amount (float)
TotalCharges    → total billed (float)
SeniorCitizen   → 1=senior, 0=not (integer)
Contract        → 0=Month-to-month, 1=One year, 2=Two year

## Why Only 5 Features?
Full one-hot encoding creates many columns (20+).
User can't input 20+ values manually.
These 5 features are most impactful for churn prediction:
- tenure → longer = less likely to churn
- MonthlyCharges → higher = more likely to churn
- Contract → month-to-month = most likely to churn

## Contract Encoding
Instead of get_dummies (creates multiple columns):
→ Used ordinal encoding (single column, 0/1/2)
Month-to-month → 0 (highest churn risk)
One year       → 1 (medium churn risk)
Two year       → 2 (lowest churn risk)

## How It Works
Step 1 → Load and clean data (same as Task 2)
Step 2 → Encode Churn (Yes=1, No=0)
Step 3 → Encode Contract as ordinal (0,1,2)
Step 4 → Train LogisticRegression on FULL dataset
Step 5 → Collect 5 inputs from user
Step 6 → Create single row DataFrame with inputs
Step 7 → model.predict() → 0 or 1
Step 8 → Print result

## User Inputs
Enter tenure (months):                          → integer
Enter Monthly Charges:                          → float
Enter Total Charges:                            → float
Senior Citizen? (1 = Yes, 0 = No):             → 0 or 1
Contract type (0=Month-to-month, 1=One year,
2=Two year):                     → 0, 1 or 2

## Output
Prediction: This customer is likely to STAY.
or
Prediction: This customer is likely to CHURN.

## Key Learnings
- Production models train on FULL data (no split needed)
- User input must match exact feature names and types model was trained on
- Ordinal encoding better than get_dummies for user input scenarios
- pd.DataFrame([[values]], columns=features) creates single row for prediction
- Always convert user inputs to correct type (int/float) before predicting

🎉 Mini Project Complete!
Task 1 → Explored dataset    → found 7 data quality issues ✅
Task 2 → Cleaned dataset     → 867 clean rows ready ✅
Task 3 → Trained model       → 69.54% accuracy ✅
Task 4 → Prediction tool     → working interactive script ✅
What you built:
An end-to-end ML pipeline — from raw messy data to a working
prediction tool that any business team member can use! 😄